In [1]:
import json
import re


# =========================
# CONFIG
# =========================
FLICKR30K_DIR = ".."
CAPTIONS_DIR = f"{FLICKR30K_DIR}/data/captions"

INPUT_FILE = f"{CAPTIONS_DIR}/captions_split.json"
OUTPUT_FILE = f"{CAPTIONS_DIR}/captions_split_clean.json"


# =========================
# STOPWORDS
# =========================
STOPWORDS: set[str] = {
    "the", "a", "an",
    "is", "are",
    "his", "her", "their", "its"
}


# =========================
# PROTECT DECIMALS
# =========================
def protect_decimals(text: str) -> tuple[str, dict[str, str]]:
    """
    Sostituisce numeri decimali con placeholder per preservarli
    durante la rimozione della punteggiatura.
    """
    mapping: dict[str, str] = {}

    def repl(match: re.Match[str]) -> str:
        token = match.group(0)
        key = f"__NUM{len(mapping)}__"
        mapping[key] = token
        return key

    text = re.sub(r"\b\d+\.\d+\b", repl, text)
    return text, mapping


# =========================
# CLEANING FUNCTION (MODIFICATA MINIMAMENTE)
# =========================
def clean_caption(
    caption: str,
    stopwords: set[str],
    drop_single_chars: bool = True
) -> str:

    caption = caption.lower()
    caption = normalize_contractions(caption)

    # 👇 SOLO AGGIUNTA
    caption, protected = protect_decimals(caption)

    caption = re.sub(r"[^\w\s]", "", caption)

    words = caption.split()

    cleaned: list[str] = []

    for i, word in enumerate(words):

        if word in protected:
            cleaned.append(protected[word])
            continue

        if word == "there":
            if i + 1 < len(words) and words[i + 1] in {"is", "are"}:
                continue

            cleaned.append(word)
            continue

        if word in stopwords:
            continue

        if drop_single_chars and len(word) == 1 and word.isalpha() and word != "i":
            continue

        cleaned.append(word)

    return " ".join(cleaned)


def normalize_contractions(caption: str) -> str:
    """
    Normalizza tutte le contrazioni Penn Treebank in modo consistente.
    Ordine importante: prima negazioni, poi altre contrazioni.
    """

    # -------------------------
    # 1. NEGATIONS (priorità alta)
    # -------------------------
    negations = {
        r"\bdo n't\b": "do not",
        r"\bdoes n't\b": "does not",
        r"\bdid n't\b": "did not",
        r"\bis n't\b": "is not",
        r"\bare n't\b": "are not",
        r"\bwas n't\b": "was not",
        r"\bwere n't\b": "were not",
        r"\bhave n't\b": "have not",
        r"\bhas n't\b": "has not",
        r"\bhad n't\b": "had not",
        r"\bcan n't\b": "cannot",
        r"\bca n't\b": "cannot",
        r"\bcould n't\b": "could not",
        r"\bshould n't\b": "should not",
        r"\bwould n't\b": "would not",
        r"\bwill n't\b": "will not",
        r"\bwo n't\b": "will not",
        r"\bmust n't\b": "must not",
    }

    for pattern, repl in negations.items():
        caption = re.sub(pattern, repl, caption)

    # -------------------------
    # 2. OTHER CONTRACTIONS
    # -------------------------
    contractions = {
        r"\bi 'm\b": "i am",
        r"\byou 're\b": "you are",
        r"\bwe 're\b": "we are",
        r"\bthey 're\b": "they are",

        r"\bi 've\b": "i have",
        r"\byou 've\b": "you have",
        r"\bwe 've\b": "we have",
        r"\bthey 've\b": "they have",

        r"\bi 'll\b": "i will",
        r"\byou 'll\b": "you will",
        r"\bhe 'll\b": "he will",
        r"\bshe 'll\b": "she will",
        r"\bwe 'll\b": "we will",
        r"\bthey 'll\b": "they will",

        r"\bi 'd\b": "i would",
        r"\byou 'd\b": "you would",
        r"\bhe 'd\b": "he would",
        r"\bshe 'd\b": "she would",
        r"\bwe 'd\b": "we would",
        r"\bthey 'd\b": "they would",

        r"\bhe 's\b": "he is",
        r"\bshe 's\b": "she is",
        r"\bit 's\b": "it is",
        r"\bthere 's\b": "there is",
        r"\bthat 's\b": "that is",
        r"\bwhat 's\b": "what is",
        r"\bwho 's\b": "who is",
    }

    for pattern, repl in contractions.items():
        caption = re.sub(pattern, repl, caption)

    return caption


# =========================
# LOAD JSON DATASET
# =========================
def load_dataset(path: str) -> dict[str, dict[str, list[str]]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# =========================
# CLEAN ALL CAPTIONS
# =========================
def clean_dataset(
    dataset: dict[str, dict[str, list[str]]],
    stopwords: set[str]
) -> dict[str, dict[str, list[str]]]:

    cleaned_dataset: dict[str, dict[str, list[str]]] = {
        "train": {},
        "val": {},
        "test": {}
    }

    for split, images in dataset.items():
        split_dict: dict[str, list[str]] = {}

        for image, captions in images.items():
            split_dict[image] = [
                clean_caption(c, stopwords)
                for c in captions
            ]

        cleaned_dataset[split] = split_dict

    return cleaned_dataset


# =========================
# SAVE JSON
# =========================
def save_json(
    dataset: dict[str, dict[str, list[str]]],
    output_file: str
) -> None:
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)


# =========================
# MAIN
# =========================
dataset = load_dataset(INPUT_FILE)

cleaned_dataset = clean_dataset(dataset, STOPWORDS)

save_json(cleaned_dataset, OUTPUT_FILE)

print("Cleaning completed ✔")
print("Saved to:", OUTPUT_FILE)

Cleaning completed ✔
Saved to: ../data/captions/captions_split_clean.json
